In [1]:
import numpy as np
import pandas as pd
import cv2
import mediapipe as mp
import csv

In [2]:
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python.vision import HandLandmarker, HandLandmarkerOptions, RunningMode

In [3]:
options = HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(
        model_asset_path='hand_landmarker.task'
    ),
    running_mode=RunningMode.IMAGE,
    num_hands=1,
    min_hand_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

- Kiểm tra xem camera có nhận được hình ảnh tay không

In [5]:
landmarker = HandLandmarker.create_from_options(options)
cap = cv2.VideoCapture(0)

print("Camera đã bật. Giơ tay vào camera, nhấn Q để thoát.")

while True:
    ret, frame = cap.read()
    frame = cv2.flip(frame, 1)

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = landmarker.detect(mp_image)

    if result.hand_landmarks:
        hand = result.hand_landmarks[0]
        row = []
        for lm in hand:
            row += [lm.x, lm.y, lm.z]

        # Hiển thị số đầu tiên để biết đang chạy đúng
        cv2.putText(frame, f'Thay tay! 63 gia tri, x0={row[0]:.2f}',
                    (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
    else:
        cv2.putText(frame, 'Khong thay tay',
                    (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)

    cv2.imshow('Thu nghiem', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
landmarker.close()
cv2.destroyAllWindows()


Camera đã bật. Giơ tay vào camera, nhấn Q để thoát.


In [4]:
LABELS = ['A', 'B', 'C']
SAMPLE_PER_LABEL = 200
OUTPUT_FILE = 'data.csv'

landmarker = HandLandmarker.create_from_options(options)
cap = cv2.VideoCapture(0)

with open(OUTPUT_FILE, 'w', newline='') as f:
    writer = csv.writer(f)
    header = ['label'] + [f'{axis}{i}' for i in range(20) for axis in ['x','y','z']]
    writer.writerow(header)

for label in LABELS:
    count = 0

    while True:
        ret, frame = cap.read()
        frame = cv2.flip(frame, 1)
        cv2.putText(frame, f'San sang: {label} | Nhan SPACE', (10,40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,200,255), 2)
        cv2.imshow("Thu thap", frame)

        if cv2.waitKey(1) & 0xFF == ord(' '):
            break
        if cv2.waitKey(1) & 0xFF == ord('q'):
            cap.release()
            cv2.destroyAllWindows() 
            exit()

    while count < SAMPLE_PER_LABEL:
        ret, frame = cap.read()
        frame = cv2.flip(frame, 1)

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = landmarker.detect(mp_image)

        if result.hand_landmarks:
            hand = result.hand_landmarks[0]
            row = [label]
            for lm in hand:
                row += [lm.x, lm.y, lm.z]
            with open(OUTPUT_FILE, 'a', newline='') as f:
                csv.writer(f).writerow(row)
            count+=1

        cv2.putText(frame, f'{label}: {count}/{SAMPLE_PER_LABEL}', (10,40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)
        cv2.imshow("Thu thap", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            cap.release()
            cv2.destroyAllWindows() 
            exit()
            
    print(f'Xong {label}!')
    
cap.release()
cv2.destroyAllWindows()

Xong A!
Xong B!
Xong C!
